In [ ]:
python <<'PY'
import duckdb

POS_PATH = "/share/storage/monade/rota/results/blocks_mar2026/positive_blocks_analysis_10d_mar2026_v3.parquet"
LISTS_PATH = "/home/monade/storage/balduf/output/03-lists.parquet"
ITEMS_PATH = "/home/monade/storage/balduf/output/03-list-items.parquet"

con = duckdb.connect()

# Positivi: una riga per account/evento di takedown
con.execute(f"""
CREATE TEMP VIEW positives AS
SELECT DISTINCT
    did,
    event_time
FROM read_parquet('{POS_PATH}');
""")

# Vere moderation lists
con.execute(f"""
CREATE TEMP VIEW modlists AS
SELECT
    did_id AS list_did_id,
    rkey AS list_rkey,
    name,
    description,
    created_at AS list_created_at
FROM read_parquet('{LISTS_PATH}')
WHERE purpose = 'app.bsky.graph.defs#modlist';
""")

# Inclusioni dei positivi in modlist nei 10 giorni prima del takedown
con.execute(f"""
CREATE TEMP VIEW positive_modlist_inclusions_10d AS
SELECT DISTINCT
    p.did,
    p.event_time,
    i.list_did_id,
    i.list_rkey,
    m.name,
    i.created_at AS inclusion_time,
    date_diff('second', i.created_at, p.event_time) / 86400.0 AS days_before_takedown
FROM positives p
JOIN read_parquet('{ITEMS_PATH}') i
    ON i.subject_id = p.did
JOIN modlists m
    ON i.list_did_id = m.list_did_id
   AND i.list_rkey = m.list_rkey
WHERE i.created_at >= p.event_time - INTERVAL 10 DAY
  AND i.created_at < p.event_time;
""")

# Una riga per positivo, con il numero di modlist ricevute nei 10 giorni
con.execute("""
CREATE TEMP VIEW positive_level AS
SELECT
    p.did,
    p.event_time,
    COUNT(DISTINCT i.list_did_id || '/' || i.list_rkey) AS n_modlists_10d,
    MIN(i.inclusion_time) AS first_modlist_inclusion_time,
    MIN(i.days_before_takedown) AS closest_inclusion_days_before_takedown,
    MAX(i.days_before_takedown) AS first_inclusion_days_before_takedown
FROM positives p
LEFT JOIN positive_modlist_inclusions_10d i
    ON p.did = i.did
   AND p.event_time = i.event_time
GROUP BY p.did, p.event_time;
""")

print("\n RISULTATI GENERALI ")
print(con.execute("""
SELECT
    COUNT(*) AS n_positive_accounts,
    SUM(n_modlists_10d > 0)::BIGINT AS n_positive_in_modlist_10d,
    ROUND(100.0 * SUM(n_modlists_10d > 0) / COUNT(*), 3) AS pct_positive_in_modlist_10d,

    COUNT(DISTINCT CASE
        WHEN n_modlists_10d > 0 THEN did
    END) AS n_distinct_positive_dids_in_modlist_10d,

    (SELECT COUNT(*) FROM positive_modlist_inclusions_10d) AS n_account_modlist_inclusions_10d,
    (SELECT COUNT(DISTINCT list_did_id || '/' || list_rkey)
     FROM positive_modlist_inclusions_10d) AS n_distinct_modlists_involved,

    ROUND(AVG(n_modlists_10d), 3) AS avg_modlists_all_positives,
    ROUND(AVG(n_modlists_10d) FILTER (WHERE n_modlists_10d > 0), 3) AS avg_modlists_only_listed_positives,
    MEDIAN(n_modlists_10d) FILTER (WHERE n_modlists_10d > 0) AS median_modlists_only_listed_positives,
    MAX(n_modlists_10d) AS max_modlists_for_one_positive
FROM positive_level;
""").df().to_string(index=False))

print("\n TEMPO TRA INCLUSIONE IN MODLIST E TAKEDOWN ")
print(con.execute("""
SELECT
    ROUND(AVG(days_before_takedown), 3) AS avg_days_before_takedown,
    ROUND(MEDIAN(days_before_takedown), 3) AS median_days_before_takedown,
    ROUND(MIN(days_before_takedown), 3) AS closest_inclusion_days_before_takedown,
    ROUND(MAX(days_before_takedown), 3) AS earliest_inclusion_days_before_takedown
FROM positive_modlist_inclusions_10d;
""").df().to_string(index=False))

print("\n DISTRIBUZIONE: NUMERO DI MODLIST PER POSITIVO ")
print(con.execute("""
SELECT
    n_modlists_10d,
    COUNT(*) AS n_positive_accounts,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM positive_level), 3) AS pct_positive_accounts
FROM positive_level
GROUP BY n_modlists_10d
ORDER BY n_modlists_10d;
""").df().to_string(index=False))

print("\n TOP 20 MODLIST PER NUMERO DI POSITIVI INSERITI NEI 10 GIORNI ")
print(con.execute("""
SELECT
    list_did_id,
    list_rkey,
    name,
    COUNT(DISTINCT did) AS n_positive_accounts_listed,
    COUNT(*) AS n_positive_inclusion_events,
    ROUND(AVG(days_before_takedown), 3) AS avg_days_before_takedown,
    ROUND(MEDIAN(days_before_takedown), 3) AS median_days_before_takedown
FROM positive_modlist_inclusions_10d
GROUP BY list_did_id, list_rkey, name
ORDER BY n_positive_accounts_listed DESC, n_positive_inclusion_events DESC
LIMIT 20;
""").df().to_string(index=False))

print("\n POSITIVI PIÙ PRESENTI IN MODLIST ")
print(con.execute("""
SELECT
    did,
    event_time,
    n_modlists_10d,
    first_modlist_inclusion_time,
    ROUND(first_inclusion_days_before_takedown, 3) AS first_inclusion_days_before_takedown,
    ROUND(closest_inclusion_days_before_takedown, 3) AS closest_inclusion_days_before_takedown
FROM positive_level
WHERE n_modlists_10d > 0
ORDER BY n_modlists_10d DESC, first_modlist_inclusion_time
LIMIT 20;
""").df().to_string(index=False))

con.close()
PY


#Account positivi totali
#47.821
#Account positivi inclusi in almeno una modlist nei 10 giorni precedenti
#929
#Percentuale di positivi inclusi in almeno una modlist
#1,943%
#DID positivi distinti inclusi in almeno una modlist
#929
#Inclusioni account–modlist osservate nei 10 giorni precedenti
#1.033
#Modlist distinte coinvolte
#73
#Numero medio di modlist per positivo, includendo gli account mai inclusi
#0,021
#Numero medio di modlist tra i soli positivi inclusi in almeno una lista
#1,090
#Mediana delle modlist tra i soli positivi inclusi
#1
#Numero massimo di modlist associate a un singolo positivo
#7



In [ ]:
#positivi presenti in almento una lista prima del takedown

python <<'PY'
import duckdb

POS_PATH = "/share/storage/monade/rota/results/blocks_mar2026/positive_blocks_analysis_10d_mar2026_v3.parquet"
LISTS_PATH = "/home/monade/storage/balduf/output/03-lists.parquet"
ITEMS_PATH = "/home/monade/storage/balduf/output/03-list-items.parquet"

con = duckdb.connect()

print("\n POSITIVI PRESENTI IN ALMENO UNA MODLIST PRIMA DEL TAKEDOWN ")

print(con.execute(f"""
WITH positives AS (
    SELECT DISTINCT
        did,
        event_time
    FROM read_parquet('{POS_PATH}')
),

positive_in_modlist_before_takedown AS (
    SELECT DISTINCT
        p.did
    FROM positives p
    JOIN read_parquet('{ITEMS_PATH}') i
        ON i.subject_id = p.did
    JOIN read_parquet('{LISTS_PATH}') l
        ON i.list_did_id = l.did_id
       AND i.list_rkey = l.rkey
    WHERE l.purpose = 'app.bsky.graph.defs#modlist'
      AND i.created_at < p.event_time
)

SELECT
    (SELECT COUNT(*) FROM positives) AS n_positive_accounts,
    (SELECT COUNT(*) FROM positive_in_modlist_before_takedown) AS n_positive_in_at_least_one_modlist_before_takedown,
    ROUND(
        100.0 * (SELECT COUNT(*) FROM positive_in_modlist_before_takedown)
        / (SELECT COUNT(*) FROM positives),
        3
    ) AS pct_positive_in_at_least_one_modlist_before_takedown;
""").df().to_string(index=False))

con.close()
PY

# POSITIVI PRESENTI IN ALMENO UNA MODLIST PRIMA DEL TAKEDOWN 
#n_positive_accounts  n_positive_in_at_least_one_modlist_before_takedown  pct_positive_in_at_least_one_modlist_before_takedown
#              47821                                                 957                                                 2.001

# ricordiamo che 20574 è il num di positivi in 00 -> pct = 3,51 %